# Projeto 3 — Interpolação e Regressão Polinomial
Este notebook implementa os conceitos da questão 1 para as questões 2 e 3 do Projeto 3 em Julia.
Ele mostra a matriz de avaliação em base canônica (Vandermonde), a base de Lagrange, e a matriz de regressão por mínimos quadrados.

## 1. Funções auxiliares
Definimos a matriz de Vandermonde para a base canônica e uma função para construir a matriz de avaliação na base de Lagrange.

In [1]:
using LinearAlgebra

function vandermonde(x::Vector{Float64}, d::Integer)
    m = length(x)
    V = zeros(Float64, m, d + 1)
    for i in 1:m, j in 1:d + 1
        V[i, j] = x[i]^(j - 1)
    end
    return V
end

function least_squares_matrix(A::AbstractMatrix{Float64})
    # A é a matriz de avaliação L_X em alguma base de P_d.
    # A solução dos mínimos quadrados seria p = R_X,d * b, onde R_X,d = pinv(A).
    return pinv(A)
end

function lagrange_basis(nodes::Vector{Float64})
    n = length(nodes)
    basis = Vector{Function}(undef, n)
    for i in 1:n
        xi = nodes[i]
        denom = prod(xi - nodes[j] for j in 1:n if j != i)
        basis[i] = x -> begin
            num = prod(x - nodes[j] for j in 1:n if j != i)
            return num / denom
        end
    end
    return basis
end

function lagrange_eval_matrix(X::Vector{Float64}, nodes::Vector{Float64})
    basis = lagrange_basis(nodes)
    m = length(X)
    k = length(nodes)
    M = zeros(Float64, m, k)
    for i in 1:m, j in 1:k
        M[i, j] = basis[j](X[i])
    end
    return M
end

lagrange_eval_matrix (generic function with 1 method)

## 2. Montando as matrizes de avaliação e regressão
Aqui simulamos um conjunto de pontos distintos `X` e construímos as matrizes `L_X` e `R_X,d` em duas bases diferentes.

In [2]:
X = collect(range(-1.0, 1.0, length = 12))
d = 5
A = vandermonde(X, d)
R = least_squares_matrix(A)
println("Matriz L_X em base canônica (Vandermonde): tamanho = ", size(A))
println("Matriz R_X,d em base canônica: tamanho = ", size(R))
println("cond(L_X) = ", cond(A))
println("cond(R_X,d) = ", cond(R))

Matriz L_X em base canônica (Vandermonde): tamanho = (12, 6)
Matriz R_X,d em base canônica: tamanho = (6, 12)
cond(L_X) = 38.81504917682817
cond(R_X,d) = 38.81504917682821


## 3. Base de Lagrange
Construímos a base de Lagrange usando os primeiros `d+1` pontos de `X`.

In [3]:
nodes = X[1:(d + 1)]
A_lagr = lagrange_eval_matrix(X, nodes)
R_lagr = least_squares_matrix(A_lagr)
println("Matriz L_X em base de Lagrange: tamanho = ", size(A_lagr))
println("cond(L_X) em Lagrange = ", cond(A_lagr))
println("cond(R_X,d) em Lagrange = ", cond(R_lagr))

Matriz L_X em base de Lagrange: tamanho = (12, 6)
cond(L_X) em Lagrange = 6072.4735321966655
cond(R_X,d) em Lagrange = 6072.4735321968155


## 4. Teste com um vetor de dados
A melhor forma de ver o efeito do condicionamento é aplicar as matrizes a um vetor de observações.

In [4]:
b = sin.(2π .* X) .+ 0.1 .* randn(length(X))
p_mono = R * b
p_lagr = R_lagr * b
println("Erro residual usando base canônica: ", norm(A * p_mono - b))
println("Erro residual usando base de Lagrange: ", norm(A_lagr * p_lagr - b))

Erro residual usando base canônica: 0.8467501197218853
Erro residual usando base de Lagrange: 0.846750119721816


## 5. Interpretação e próximos passos
- `L_X` em base canônica é a matriz de Vandermonde.
- `R_X,d` é a pseudo-inversa de `L_X`, que representa a regressão de mínimos quadrados.
- A base de Lagrange muda a forma de `L_X`, e isso pode afetar o condicionamento.

### Para avançar
1. Varie `m` e `d` e veja como `cond(A)` e `cond(R)` mudam.
2. Compare o comportamento usando `X` equiespaçados e `X` de Chebyshev.
3. Adicione um gráfico com `Plots` para mostrar o crescimento do condicionamento com `d`.